#Cleaning Data

First, let's take a look at our raw data

In [1]:
import pandas as pd
plrty = pd.read_csv("Polarity_rawData_50.csv")
plrty.head()

,id,Title,Text,Reads,Likes,Shares,Hearts,Date Published
0,1,Interview: Struggles in Learning English and H...,Interview: Struggles in Learning English and H...,198,12,10,1,16/01/2024
1,2,Strength in Struggle,Strength in Struggle\nImage\n\nLife—it's tough...,227,13,16,2,24/01/2024
2,3,付费自习室的兴起，预示着什么?,付费自习室的兴起，预示着什么?\nImage\n\n付费自习室的兴起，预示着什么？\n\nT...,319,10,13,2,31/01/2024
3,4,Focus,Focus\nImage\n\nWere you ever given a task tha...,145,8,7,3,06/02/2024
4,5,Hamilton 汉密尔顿,Hamilton 汉密尔顿\n\n\nHamilton--an epic of freedo...,320,10,12,1,13/02/2024


Right now, text has some words that aren't a part of the article such as images replace by the word "Image".

In [2]:
plrty["Text"]

,Text
0,Interview: Struggles in Learning English and H...
1,Strength in Struggle\nImage\n\nLife—it's tough...
2,付费自习室的兴起，预示着什么?\nImage\n\n付费自习室的兴起，预示着什么？\n\nT...
3,Focus\nImage\n\nWere you ever given a task tha...
4,Hamilton 汉密尔顿\n\n\nHamilton--an epic of freedo...
5,Social Anxiety 社交焦虑\nImage\n\nReducing social ...
6,The Catcher in the Rye 麦田里的守望者\nImage\n\nThe C...
7,Learning Through Imitation 在模仿中学习\nImage\n\nHa...
8,Interview: Advice on Career Picking and Prepar...
9,Falling in Love with...Yourself? 爱上...你自己?\nIm...


Before removing these "Image"s, it is helpful to count how many there are for future analysis. To account for instances where a valid instance of "Image" does not correspond to a picture, we will count the number of "Image"s before and after removing them.

In [3]:
#counts all instances of "Image"
image_before = plrty["Text"].str.count("Image")

#remove "Images" only when in its own line and put cleaned text in a new column
plrty["cleanText"] = (
    plrty["Text"].str.replace(r"(?m)^(?:Image)+(?:\n|$)", "", regex=True)
)

#counts instances of "Image" in the article text
image_after = plrty["cleanText"].str.count("Image")

#add a new column for number of images
plrty["num_images"] = image_before - image_after
print(plrty["num_images"])


0      9
1      3
2      3
3      3
4      3
5      3
6      3
7      3
8      7
9      3
10     3
11     3
12     5
13     4
14     3
15    11
16     5
17     3
18     3
19     3
20     8
21     3
22     3
23     4
24     3
25     2
26     2
27     3
28     3
29     3
30     0
31     4
32     7
33     7
34     9
35     5
36     3
37     7
38     6
39     6
40     6
41     8
42     7
43     8
44     3
45     5
46     6
47     3
48    12
49     8
Name: num_images, dtype: int64


There are some empty lines at the end of some articles

In [4]:
#remove empty lines at the end of article
plrty["cleanText"] = plrty["cleanText"].str.rstrip()


Let's check an article

In [5]:
print(plrty.loc[1, "cleanText"])

Strength in Struggle

Life—it's tough, draining, and a shared experience for us all. Many wake up and fall asleep with troubled thoughts and heavy hearts, grappling with struggles in school, family, friendships, or love. These hardships might seem like they drag us down, hindering our progress. But is that really the case? Do the hardships we face tear us apart and thwart our goals? No. Seriously, no. While most would argue against it, consider this: Can you achieve greatness without hardships and effort? And would it still be fulfilling to achieve something without lifting your finger? Absolutely not, so reconsider.

 

生活——对我们所有人来说都是艰难的，令人筋疲力尽的。许多人入睡时都带着烦恼的想法和沉重的心情，醒来时亦是如此，在学校、家庭、友谊或爱情中挣扎。这些困难看起来似乎是我们的累赘，但事实真是如此吗？我们面临的困难是否会使我们粉身碎骨，并最终阻碍我们实现梦想？不，说真的，并非如此。虽然大多数人会下意识地反对这一点，但请考虑一下：不经历困难和努力，你能取得伟大成就吗？不费吹灰之力就能实现的某些目标能让你有成就感吗？显然不行，所以，请重新思考一下这些煎熬的意义。

 

In life, the battles we conquer and endure are countless and will likely remain endless. Should we really allow these obstacles stop us fro

So far so good. For a deeper analysis, we may want to get data on title length and text length

In [6]:
plrty["titleLength"] = plrty["Title"].str.len()
print(plrty["titleLength"])

0     59
1     20
2     15
3      5
4     13
5     19
6     30
7     33
8     51
9     42
10    13
11    29
12    57
13    34
14    36
15    24
16    35
17    47
18    62
19    57
20    41
21    24
22    42
23    25
24    22
25    21
26    47
27    26
28    24
29    13
30    46
31    35
32    39
33    49
34    14
35    19
36    23
37    42
38     7
39    32
40    25
41    48
42    43
43    33
44    18
45    33
46    27
47    16
48    10
49    49
Name: titleLength, dtype: int64


To analyze the effect on the type of language on the number reads, I classified titles into four categories: Bilingual (English text comes first, EC), Bilingual(Chinese text comes first, CE), English (E), and Chinese(C)

In [7]:
def classify_title(title):
    chinese = next((i for i, char in enumerate(title) if "\u4e00" <= char <= "\u9fff"), None)
    english = next((i for i, char in enumerate(title) if char.isascii() and char.isalpha()), None)
    if chinese is None:
      return "E"
    elif english is None:
      return "C"
    elif english < chinese:
      return "EC"
    else:
      return "CE"

plrty["Title_Language"] = plrty["Title"].apply(classify_title)
print(plrty["Title_Language"])

0      E
1      E
2      C
3      E
4     EC
5     EC
6     EC
7     EC
8      E
9     EC
10    EC
11    EC
12     E
13     E
14     E
15     E
16     E
17    EC
18     E
19     E
20    EC
21    EC
22    CE
23    EC
24    EC
25    CE
26    EC
27     C
28    EC
29    EC
30    EC
31     E
32     E
33     E
34     C
35    EC
36     E
37     E
38    EC
39    EC
40     E
41     E
42    CE
43     E
44     C
45    CE
46     E
47     E
48     C
49     E
Name: Title_Language, dtype: object


In [8]:
#Gets text length
plrty["textLength"] = plrty["cleanText"].str.len()
print(plrty["textLength"])

0     2841
1     4453
2     6755
3     4274
4     5935
5     5317
6     5921
7     4786
8     5336
9     4499
10    7109
11    4853
12    6194
13    4185
14    6955
15    3769
16    1472
17    3667
18    3611
19    4290
20    1706
21    2778
22    2576
23    4167
24    7704
25    3616
26    3804
27    5059
28    4673
29    2581
30    3806
31    5350
32    8456
33    5976
34    2955
35    3897
36    6004
37    4323
38    7513
39    3765
40    5665
41    9753
42    4272
43    7626
44    3824
45    4661
46    3830
47    2991
48    8196
49    9861
Name: textLength, dtype: int64


Finally, our table looks like this

In [9]:
plrty.head()

,id,Title,Text,Reads,Likes,Shares,Hearts,Date Published,cleanText,num_images,titleLength,Title_Language,textLength
0,1,Interview: Struggles in Learning English and H...,Interview: Struggles in Learning English and H...,198,12,10,1,16/01/2024,Interview: Struggles in Learning English and H...,9,59,E,2841
1,2,Strength in Struggle,Strength in Struggle\nImage\n\nLife—it's tough...,227,13,16,2,24/01/2024,"Strength in Struggle\n\nLife—it's tough, drain...",3,20,E,4453
2,3,付费自习室的兴起，预示着什么?,付费自习室的兴起，预示着什么?\nImage\n\n付费自习室的兴起，预示着什么？\n\nT...,319,10,13,2,31/01/2024,付费自习室的兴起，预示着什么?\n\n付费自习室的兴起，预示着什么？\n\nThe Rise...,3,15,C,6755
3,4,Focus,Focus\nImage\n\nWere you ever given a task tha...,145,8,7,3,06/02/2024,Focus\n\nWere you ever given a task that you s...,3,5,E,4274
4,5,Hamilton 汉密尔顿,Hamilton 汉密尔顿\n\n\nHamilton--an epic of freedo...,320,10,12,1,13/02/2024,Hamilton 汉密尔顿\n\n\nHamilton--an epic of freedo...,3,13,EC,5935


In [10]:
plrty.to_csv("Polarity_cleaned_50.csv", index=False)